# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata object provides attributes for dataset description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each component in the Croissant schema is uniquely identified by an `@id`. Here, we list out the available record sets, their fields, and their respective IDs.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, datatype: {field.data_type})")
    print()
    record_set_ids.append(rs.id)
# For demonstration, set a main record set variable
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, all record sets are loaded into a dictionary of DataFrames, keyed by their `@id`.

In [ ]:
# Extract data from each record set (@id-based)
dataframes = {}
for record_set in dataset.record_sets:
    print(f"Loading records from RecordSet: {record_set.name} (@id: {record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"  Loaded {len(df)} records.")
        print(f"  Columns: {df.columns.tolist()}\n")
    else:
        print("  No records found.\n")
# Pick first non-empty dataframe as default for further EDA
record_set_for_demo = None
for rs_id, df in dataframes.items():
    if not df.empty:
        record_set_for_demo = rs_id
        break
if record_set_for_demo is not None:
    print(f"Using RecordSet ID: {record_set_for_demo} for EDA")
    print("Columns in this DataFrame:", dataframes[record_set_for_demo].columns.tolist())
    display(dataframes[record_set_for_demo].head())
else:
    print("No data found in any RecordSet.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping by key attributes using `@id`s.

Below, we select a numeric field for filtering and normalization, and a group field for aggregation. All fields are referenced using their `@id`.

In [ ]:
# Select a numeric field and group field by @id
# You may replace these IDs below with your dataset's relevant field IDs from the earlier overview
numeric_field_id = None
group_field_id = None
df = dataframes.get(record_set_for_demo, pd.DataFrame())
if not df.empty:
    # Try auto-detect numeric fields
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    # Try auto-detect string/object fields for grouping
    group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
    if group_candidates:
        group_field_id = group_candidates[0]

    print(f"Numeric field chosen (by @id/column): {numeric_field_id}")
    print(f"Group-by field chosen (by @id/column): {group_field_id}")

    # EDA: filter, normalize, group
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by '{group_field_id}':")
            display(grouped_df.head())
else:
    print("Data not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Fields should be referenced by their `@id` (column name in the DataFrame).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = dataframes.get(record_set_for_demo, pd.DataFrame())

if not df.empty and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(10,6))
    if (group_field_id in df.columns):
        top_groups = df[group_field_id].value_counts().nlargest(10).index
        sns.boxplot(
            data=df[df[group_field_id].isin(top_groups)],
            x=group_field_id,
            y=numeric_field_id
        )
        plt.title(f"{numeric_field_id} by {group_field_id} (Top 10)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Cannot plot: Data, numeric field, or group field missing.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset metadata using the Croissant schema and listed available record sets and fields by their `@id`.
- Data extraction and EDA was demonstrated for the primary record set.
- Basic EDA steps such as filtering, normalization, grouping, and visualizations were performed using field IDs for reproducibility.
- Use the provided field and record set `@id`s to extend this analysis for other parts of the dataset.

For more details on the Croissant specification, see: [MLCommons Croissant](https://mlcommons.org/croissant/).